In [1]:
import pandas as pd
import numpy as np
import sumolib
from modules.sumo import TLIncomingCorridors,GetRouteGreenTime,create_initial_tl_df,get_upper_bounds
import torch
from modules.inverse_cdf import InverseCDF
import matplotlib.pyplot as plt

In [2]:
all_routes_df=None

route_files_list=["./Ingolstadt Network/InTAS_016.rou.xml","./Ingolstadt Network/InTAS_017.rou.xml","./Ingolstadt Network/InTAS_018.rou.xml"]
for route_file in route_files_list:
    route_df=pd.read_xml(route_file,iterparse={"route":["edges"]})
    route_df=route_df.rename(columns={"edges":"route_edges"})
    if all_routes_df is None:
        all_routes_df=route_df
    else:
        all_routes_df=pd.concat([all_routes_df,route_df],axis=0,ignore_index=True)

In [3]:
additional_gt_xml="./Ingolstadt Network/additional_gt.xml"
net_xml="./Ingolstadt Network/ingolstadt.net.xml"


corridors_df=torch.load("./corridors_savings/corridors_df.pt",weights_only=False)
complete_corridors_df=torch.load("./corridors_savings/complete_corridors_connexions_df.pt",weights_only=False)

initial_tl_df=create_initial_tl_df(additional_gt_xml)

route_green_time_accessor=GetRouteGreenTime(net_file=net_xml,initial_tl_df=initial_tl_df)

In [13]:
corridors_df[["tl","route_str_str","route_id","route_last_edge","route_demand"]].iloc[217]

tl                                        cluster_30503255_335525526
route_str_str      29238188#0 -25665374 29027727#6 25665375#0 320...
route_id                                                         492
route_last_edge                                             54858974
route_demand                                                       3
Name: 492, dtype: object

In [16]:
corridors_df.iloc[217]["route_str_str"]

'29238188#0 -25665374 29027727#6 25665375#0 32007121 25901651#3 29027727#9 152077572#0 152077572#2 152077572#2.79 54858974'

In [10]:
len(corridors_df["route_id"].unique())

2134

In [3]:
corridors_df=route_green_time_accessor.get_route_green_time(corridors_df)

In [15]:
corridors_df[["tl","route_id","route_last_edge","route_green_phases","route_green_link_indices","main_green_phase","main_green_phase_duration"]]

,tl,route_id,route_last_edge,route_green_phases,route_green_link_indices,main_green_phase,main_green_phase_duration
0,cluster_277739062_39732718_89129091_89129092,0,-102818928#0.78,[GGGrrrr],"[0, 1]",GGGrrrr,37
1,cluster_277739062_39732718_89129091_89129092,1,-201969576#0,"[rrGGGGg, rryyyyg, rrrrrGG]","[5, 6]",rrGGGGg,38
2,cluster_277739062_39732718_89129091_89129092,2,61940355,"[rrGGGGg, GGGrrrr]","[2, 3, 4]",rrGGGGg,38
3,267380207,3,-32201172,[rrGGrrrGg],"[7, 8]",rrGGrrrGg,27
4,267380207,4,145069373#0,[rrrrGGGrr],"[4, 5, 6]",rrrrGGGrr,27
...,...,...,...,...,...,...,...
781,cluster_475944687_475944688_480162640,781,39649524#0,[GGrrrrr],"[0, 1]",GGrrrrr,16
782,cluster_475944687_475944688_480162640,782,27453586#0,"[rrGGGGr, rryyGGr, rrrrGGG]","[4, 5, 6]",rrGGGGr,40
783,cluster_475944687_475944688_480162640,783,380271165#0,[rrGGGGr],"[2, 3]",rrGGGGr,40
784,2033094557,784,30131201#1,"[GGGrr, GrrrG]","[0, 1]",GGGrr,26


In [7]:
corridors_df.columns

Index(['tl', 'route', 'route_str', 'route_str_str', 'route_last_edge',
       'route_first_edge', 'route_length_ff_tt', 'route_sum_length',
       'route_sum_ff_tt', 'route_ff_speed', 'origin', 'route_n_edges',
       'route_demand', 'tl_demand', 'tl_n_routes', 'tl_n_edges', 'add_info',
       'n_previous_tl', 'n_previous_routes', 'n_previous_edges',
       'previous_demand', 'next_info', 'direct_next_edges', 'direct_next_tls',
       'from_info', 'direct_from_edges', 'direct_previous_tls',
       'direct_next_demand', 'direct_previous_demand',
       'direct_from_n_junction_inc_lanes', 'route_tl_id', 'route_id',
       'route_weight', 'green_phases_info', 'route_green_phases',
       'route_green_link_indices', 'main_green_phase_info', 'main_green_phase',
       'main_green_phase_duration'],
      dtype='str')

In [16]:
df_gt=pd.read_csv("./gt_data/df_gt_InTAS.csv")

In [18]:
df_gt.columns

Index(['Unnamed: 0', 'route_id', 'route_average_traveltime', 'tl', 'route',
       'route_str', 'route_str_str', 'route_last_edge', 'route_first_edge',
       'route_length_ff_tt', 'route_sum_length', 'route_sum_ff_tt',
       'route_ff_speed', 'origin', 'route_n_edges', 'route_demand',
       'tl_demand', 'tl_n_routes', 'tl_n_edges', 'add_info', 'n_previous_tl',
       'n_previous_routes', 'n_previous_edges', 'previous_demand', 'next_info',
       'direct_next_edges', 'direct_next_tls', 'from_info',
       'direct_from_edges', 'direct_previous_tls', 'direct_next_demand',
       'direct_previous_demand', 'direct_from_n_junction_inc_lanes',
       'route_tl_id', 'route_weight'],
      dtype='str')

In [19]:
len(df_gt)

725

In [20]:
len(df_gt["route_id"].unique())

725

In [9]:
additional_gt_xml="./Ingolstadt Network/additional_gt.xml"
net_xml="./Ingolstadt Network/ingolstadt.net.xml"

df_gt=pd.read_csv("./gt_data/df_gt_InTAS.csv")


corridors_df=torch.load("./corridors_savings/corridors_df.pt",weights_only=False)
complete_corridors_df=torch.load("./corridors_savings/complete_corridors_connexions_df.pt",weights_only=False)

initial_tl_df=create_initial_tl_df(additional_gt_xml)

route_green_time_accessor=GetRouteGreenTime(net_file=net_xml,initial_tl_df=initial_tl_df)

df_gt=route_green_time_accessor.get_route_green_time(df_gt)
print(df_gt.shape)
print(len(df_gt["route_id"].unique()))
print(df_gt)
df_gt=pd.merge(df_gt,initial_tl_df[["tl","state","tl_cumulative_constant_duration"]].rename(columns={"state":"main_green_phase"}),on=["tl","main_green_phase"],how="left")
df_gt=df_gt.drop_duplicates(subset=["route_str_str"])
print(df_gt.shape)


(725, 43)
725
     Unnamed: 0  route_id  route_average_traveltime  \
0             0         0                 38.335185   
1             1         1                101.087500   
2             2         2                557.996848   
3             3         3                 60.900000   
4             4         4                148.200000   
..          ...       ...                       ...   
720         720       781                117.547945   
721         721       782                 67.973316   
722         722       783                 19.530253   
723         723       784                 50.992029   
724         724       785                 43.505000   

                                               tl  \
0    cluster_277739062_39732718_89129091_89129092   
1    cluster_277739062_39732718_89129091_89129092   
2    cluster_277739062_39732718_89129091_89129092   
3                                       267380207   
4                                       267380207   
..     

In [10]:
df_gt.columns

Index(['Unnamed: 0', 'route_id', 'route_average_traveltime', 'tl', 'route',
       'route_str', 'route_str_str', 'route_last_edge', 'route_first_edge',
       'route_length_ff_tt', 'route_sum_length', 'route_sum_ff_tt',
       'route_ff_speed', 'origin', 'route_n_edges', 'route_demand',
       'tl_demand', 'tl_n_routes', 'tl_n_edges', 'add_info', 'n_previous_tl',
       'n_previous_routes', 'n_previous_edges', 'previous_demand', 'next_info',
       'direct_next_edges', 'direct_next_tls', 'from_info',
       'direct_from_edges', 'direct_previous_tls', 'direct_next_demand',
       'direct_previous_demand', 'direct_from_n_junction_inc_lanes',
       'route_tl_id', 'route_weight', 'green_phases_info',
       'route_green_phases', 'route_green_link_indices',
       'main_green_phase_info', 'main_green_phase',
       'main_green_phase_duration', 'main_green_phase_to_change',
       'main_green_phase_initial_duration', 'tl_cumulative_constant_duration'],
      dtype='str')

In [2]:
additional_gt_xml="./Ingolstadt Network/additional_gt.xml"
net_xml="./Ingolstadt Network/ingolstadt.net.xml"

df_gt=pd.read_csv("./gt_data/df_gt_InTAS.csv")
initial_tl_df=create_initial_tl_df(additional_gt_xml)
route_green_time_accessor=GetRouteGreenTime(net_file=net_xml,initial_tl_df=initial_tl_df)
x_dim=98
lower_bounds=[5]*x_dim
upper_bounds=get_upper_bounds(initial_tl_df)
bounds=np.array([lower_bounds,upper_bounds])
cdf_inversor=InverseCDF(df_gt=df_gt,initial_tl_df=initial_tl_df,bounds=bounds,route_green_time_accessor=route_green_time_accessor)

In [3]:
g_vec=cdf_inversor.sample_g_vec()

In [6]:
tl_ids=initial_tl_df["tl"].unique()

def plot_pdf(tl,bounds):
    print(bounds[0])
    print(bounds[1])
    z=cdf_inversor.compute_Z(bounds,tl)
    g=np.arange(bounds[0],bounds[1])
    vectorized_pdf=np.vectorize(cdf_inversor.tl_normalized_pdf)
    pdf_values=vectorized_pdf(g,tl,z)
    return pdf_values

pdf_values=plot_pdf(tl_ids[0],bounds[:,0])

5
68


In [5]:
pdf_values

array([0.01372223, 0.01421225, 0.01480982, 0.01536341, 0.01583324,
       0.01621786, 0.0165278 , 0.01677595, 0.01697427, 0.01713285,
       0.01725986, 0.0173618 , 0.0174438 , 0.01750989, 0.01756324,
       0.01760636, 0.01764122, 0.01766941, 0.01769219, 0.01771055,
       0.01772531, 0.01773712, 0.01774651, 0.01775391, 0.01775967,
       0.01776408, 0.01776736, 0.0177697 , 0.01777126, 0.01777216,
       0.01777249, 0.01777234, 0.01777176, 0.01777081, 0.0177695 ,
       0.01776786, 0.01776589, 0.01776359, 0.01776093, 0.01775789,
       0.01775442, 0.01775046, 0.01774593, 0.01774072, 0.0177347 ,
       0.0177277 , 0.01771948, 0.01770976, 0.01769814, 0.0176841 ,
       0.01766695, 0.01764569, 0.01761895, 0.01758469, 0.01753992,
       0.01747995, 0.01739727, 0.01727915, 0.01710276, 0.01682405,
       0.01635008, 0.01546257, 0.01359972])

In [7]:
plt.plot(np.arange(5,68),pdf_values)

: 

In [3]:
plt.plot([1, 2, 3], [1, 2, 3])
plt.show()

: 